In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [177]:
import pandas as pd
from preprocess import fromat_vendor_name, format_shop_category_name, process_description, process_vendor_name, process_shop_category_name, extract_description
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.metrics import f1_score
from sklearn.decomposition import TruncatedSVD

In [58]:
df = pd.read_csv('train.tsv', sep='\t')

In [80]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['category_id', 'department_id'], axis=1),
                                                    df[['category_id', 'department_id']],
                                                    test_size=0.2, random_state=42)

In [81]:
top_vendor_names = fromat_vendor_name(X_train)["vendor_name"].value_counts().head(11).index.to_list()
top_category_names = format_shop_category_name(X_train)["shop_category_name"].value_counts().head(21).index.to_list()

In [140]:
def preprocess(df: pd.DataFrame):
    df = df.drop("vendor_code", axis=1)
    df = process_description(df)
    df = process_vendor_name(df, top_vendor_names)
    df = process_shop_category_name(df, top_category_names)
    return df

In [246]:
X_train_pr = preprocess(X_train)
X_test_pr = preprocess(X_test)

In [ ]:
char_tfidf = TfidfVectorizer(analyzer="char").fit(X_train_pr["description"])
# char_tfidf_svd = TruncatedSVD(83).fit(char_tfidf.transform(X_train_pr["description"]))
word_tfidf = TfidfVectorizer(analyzer="word").fit(X_train_pr["description"]) # MAX_FEATURES ARGUMENT
word_tfidf_svd = TruncatedSVD(2048).fit(word_tfidf.transform(X_train_pr["description"]))
char_count = CountVectorizer(analyzer="char").fit(X_train_pr["description"])
# char_count_svd = TruncatedSVD(83).fit(char_count.transform(X_train_pr["description"]))
word_count = CountVectorizer(analyzer="word").fit(X_train_pr["description"])
word_count_svd = TruncatedSVD(2048).fit(word_count.transform(X_train_pr["description"]))

In [245]:
word_tfidf_svd.explained_variance_ratio_.sum(), word_count_svd.explained_variance_ratio_.sum()

(np.float64(0.654740348128731), np.float64(0.886662454519565))

In [247]:
X_train_pr = extract_description(X_train_pr, {
    "char_tfidf": char_tfidf,
    "word_tfidf": word_tfidf,
    "char_count": char_count,
    "word_count": word_count,
}, [
    None,
    word_tfidf_svd,
    None,
    word_count_svd
])

X_test_pr = extract_description(X_test_pr, {
    "char_tfidf": char_tfidf,
    "word_tfidf": word_tfidf,
    "char_count": char_count,
    "word_count": word_count,
}, [
    None,
    word_tfidf_svd,
    None,
    word_count_svd
])

In [248]:
rfs = RandomForestClassifier()

In [249]:
X_train_pr_n = X_train_pr.select_dtypes(include=np.number)
X_test_pr_n = X_test_pr.select_dtypes(include=np.number)

In [ ]:
rfs.fit(X_train_pr_n, y_train.sort_index()["department_id"])

In [ ]:
y_pred = rfs.predict(X_test_pr_n)

In [ ]:
f1_score(y_test.sort_index()["department_id"], y_pred, average="weighted")

0.4530314801286541

0.14978467104694057